# Claim-Frequency model — 2026 re-run

This notebook re-runs the **2025 winning frequency model** ([CAA / ENS Challenge
Data #161](https://challengedata.ens.fr/challenges/161)) with the **current AutoCarver** (see `pyproject.toml` for the pinned floor).

Everything before the carving step — data loading, the `0/1/2+` target, the
stratified split, inverse-frequency weights, feature engineering and the column
typing — is **identical to `frequency_model.ipynb`** (cells reused verbatim) so
the only thing that changes is the AutoCarver step. That keeps the comparison
honest.

**What changes in 2026:**
1. **`ProcessingConfig(n_jobs=...)`** — per-feature carving runs across a process
   pool. Timed below.
2. **`OrdinalCarver`** instead of `MulticlassCarver` — the claim-count target
   `0 < 1 < 2+` is **ordinal**, so we carve with **Kendall's Tau-c** rather than
   nominal multiclass association. Note the rename: in 7.0.5 `MulticlassCarver`
   carved *one-vs-rest* (several carved columns per raw feature); that behaviour
   is `OneVsRestCarver` today.
3. **`min_freq_alpha`** — bin frequencies are gated by a **Wilson-score CI**, so
   thin bins are merged on statistical grounds (relevant for the rare-claim tail).
4. **Selectors** take a `SelectionConfig` and a single `n_best_features` **total**
   budget, split evenly across feature types.

## Loading data & target  *(reused from 2025)*

In [1]:
import pandas as pd

data_path = "../data/"

# loading x_train
data = pd.read_csv(data_path + "train_input_Z61KlZo.csv", low_memory=False)
data.set_index("ID", inplace=True)
print("x_train", data.shape)

# loading target
target = pd.read_csv(data_path + "train_output_DzPxaPY.csv", low_memory=False)
target.set_index("ID", inplace=True)
print("y_train", target.shape)

# joining x_train and y_train
data = data.join(target.drop("ANNEE_ASSURANCE", axis=1))
print("data", data.shape)

x_train (383610, 373)
y_train (383610, 4)


data (383610, 376)


In [2]:
target_col = "TARGET"
data[target_col] = (data["FREQ"] * data["ANNEE_ASSURANCE"]).astype(int)
data[target_col].value_counts(normalize=False).sort_index()

TARGET
0    381061
1      2458
2        87
3         2
4         1
5         1
Name: count, dtype: int64

## Stratified sampling & inverse-frequency weights  *(reused from 2025)*

In [3]:
import numpy as np

from collections import Counter
from sklearn.model_selection import train_test_split
from utils.data_toolkit import collapse_count

# Compute class frequencies
class_counts = Counter(collapse_count(data[target_col]))
total_samples = len(data[target_col])

# Compute inverse frequency class weights
class_weights = {
    cls: total_samples / (len(class_counts) * count)
    for cls, count in class_counts.items()
}
print("Class weights:", class_weights)

# Assign sample weights based on target values
weights = np.array([class_weights[label] for label in collapse_count(data[target_col])])

# Train-test split
x_train, x_dev, y_train, y_dev, w_train, w_dev = train_test_split(
    data,
    data[target_col],
    weights,
    test_size=0.2,
    random_state=42,
    stratify=collapse_count(data[target_col]),
)
print("y_train mean", y_train.mean(), " observations:", x_train.shape[0])
print("y_dev mean", y_dev.mean(), " observations:", x_dev.shape[0])

Class weights: {0: 0.3355630725789309, 1: 52.0219690805533, 2: 1405.1648351648353}


y_train mean 0.006895023591668622  observations: 306888
y_dev mean 0.006921091733792133  observations: 76722


## Feature engineering  *(reused from 2025)*

In [4]:
from utils.data_toolkit import Processor

proc = Processor()
x_train = proc.fit_transform(x_train)
x_dev = proc.transform(x_dev)
data = proc.transform(data)

## Column typing  *(reused from 2025)*

In [ ]:
from AutoCarver import Features

# sorting columns per type
features = Features.from_dataframe(x_train)

# getting ordinal columns
ordinals = [
    "NB_CASERNES", "BDTOPO_BAT_MAX_HAUTEUR", "HAUTEUR_MAX", "HAUTEUR", "BDTOPO_BAT_MAX_HAUTEUR_MAX", "MEN_SURF", "IND_SNV", "IND_INC", "IND_Y9", "IND_0_Y1", "IND", "LOG_SOC", "LOG_INC", "LOG_APA3", "LOG_AVA1", "MEN_MAIS", "MEN_COLL", "MEN_FMP", "MEN_PROP", "MEN_PAUV", "MEN", "COEFASS", "DISTANCE_111", "DISTANCE_112", "DISTANCE_121", "DISTANCE_122", "DISTANCE_123", "DISTANCE_124", "DISTANCE_131", "DISTANCE_132", "DISTANCE_133", "DISTANCE_141", "DISTANCE_142", "DISTANCE_211", "DISTANCE_212", "DISTANCE_213", "DISTANCE_221", "DISTANCE_222", "DISTANCE_223", "DISTANCE_231", "DISTANCE_242", "DISTANCE_243", "DISTANCE_244", "DISTANCE_311", "DISTANCE_312", "DISTANCE_313", "DISTANCE_321", "DISTANCE_322", "DISTANCE_323", "DISTANCE_324", "DISTANCE_331", "DISTANCE_332", "DISTANCE_333", "DISTANCE_334", "DISTANCE_335", "DISTANCE_411", "DISTANCE_412", "DISTANCE_421", "DISTANCE_422", "DISTANCE_423", "DISTANCE_511", "DISTANCE_512", "DISTANCE_521", "DISTANCE_522", "DISTANCE_523", "PROPORTION_11", "PROPORTION_12", "PROPORTION_13", "PROPORTION_14", "PROPORTION_21", "PROPORTION_22", "PROPORTION_23", "PROPORTION_24", "PROPORTION_31", "PROPORTION_32", "PROPORTION_33", "PROPORTION_41", "PROPORTION_42", "PROPORTION_51", "PROPORTION_52", "MEN_1IND", "MEN_5IND", "LOG_A1_A2", "LOG_A2_A3", "IND_Y1_Y2", "IND_Y2_Y3", "IND_Y3_Y4", "IND_Y4_Y5", "IND_Y5_Y6", "IND_Y6_Y7", "IND_Y7_Y8", "IND_Y8_Y9", "DISTANCE_1", "DISTANCE_2", "ALTITUDE_1", "ALTITUDE_2", "ALTITUDE_3", "ALTITUDE_4", "ALTITUDE_5", "NBJTX25_MM_A", "NBJTX25_MMAX_A", "NBJTX25_MSOM_A", "NBJTX0_MM_A", "NBJTX0_MMAX_A", "NBJTX0_MSOM_A", "NBJTXI27_MM_A", "NBJTXI27_MMAX_A", "NBJTXI27_MSOM_A", "NBJTXS32_MM_A", "NBJTXS32_MMAX_A", "NBJTXS32_MSOM_A", "NBJTXI20_MM_A", "NBJTXI20_MMAX_A", "NBJTXI20_MSOM_A", "NBJTX30_MM_A", "NBJTX30_MMAX_A", "NBJTX30_MSOM_A", "NBJTX35_MM_A", "NBJTX35_MMAX_A", "NBJTX35_MSOM_A", "NBJTN10_MM_A", "NBJTN10_MMAX_A", "NBJTN10_MSOM_A", "NBJTNI10_MM_A", "NBJTNI10_MMAX_A", "NBJTNI10_MSOM_A", "NBJTN5_MM_A", "NBJTN5_MMAX_A", "NBJTN5_MSOM_A", "NBJTNS25_MM_A", "NBJTNS25_MMAX_A", "NBJTNS25_MSOM_A", "NBJTNI15_MM_A", "NBJTNI15_MMAX_A", "NBJTNI15_MSOM_A", "NBJTNI20_MM_A", "NBJTNI20_MMAX_A", "NBJTNI20_MSOM_A", "NBJTNS20_MM_A", "NBJTNS20_MMAX_A", "NBJTNS20_MSOM_A", "NBJTMS24_MM_A", "NBJTMS24_MMAX_A", "NBJTMS24_MSOM_A", "TAMPLIAB_VOR_MM_A", "TAMPLIAB_VOR_MMAX_A", "TAMPLIM_VOR_MM_A", "TAMPLIM_VOR_MMAX_A", "TM_VOR_MM_A", "TM_VOR_MMAX_A", "TMM_VOR_MM_A", "TMM_VOR_MMAX_A", "TMMAX_VOR_MM_A", "TMMAX_VOR_MMAX_A", "TMMIN_VOR_MM_A", "TMMIN_VOR_MMAX_A", "TN_VOR_MM_A", "TN_VOR_MMAX_A", "TNAB_VOR_MM_A", "TNAB_VOR_MMAX_A", "TNMAX_VOR_MM_A", "TNMAX_VOR_MMAX_A", "TX_VOR_MM_A", "TX_VOR_MMAX_A", "TXAB_VOR_MM_A", "TXAB_VOR_MMAX_A", "TXMIN_VOR_MM_A", "TXMIN_VOR_MMAX_A", "NBJFF10_MM_A", "NBJFF10_MMAX_A", "NBJFF10_MSOM_A", "NBJFF16_MM_A", "NBJFF16_MMAX_A", "NBJFF16_MSOM_A", "NBJFF28_MM_A", "NBJFF28_MMAX_A", "NBJFF28_MSOM_A", "NBJFXI3S10_MM_A", "NBJFXI3S10_MMAX_A", "NBJFXI3S10_MSOM_A", "NBJFXI3S16_MM_A", "NBJFXI3S16_MMAX_A", "NBJFXI3S16_MSOM_A", "NBJFXI3S28_MM_A", "NBJFXI3S28_MMAX_A", "NBJFXI3S28_MSOM_A", "NBJFXY8_MM_A", "NBJFXY8_MMAX_A", "NBJFXY8_MSOM_A", "NBJFXY10_MM_A", "NBJFXY10_MMAX_A", "NBJFXY10_MSOM_A", "NBJFXY15_MM_A", "NBJFXY15_MMAX_A", "NBJFXY15_MSOM_A", "FFM_VOR_MM_A", "FFM_VOR_MMAX_A", "FXI3SAB_VOR_MM_A", "FXI3SAB_VOR_MMAX_A", "FXIAB_VOR_MM_A", "FXIAB_VOR_MMAX_A", "FXYAB_VOR_MM_A", "FXYAB_VOR_MMAX_A", "FFM_VOR_COM_MM_A_Y", "FFM_VOR_COM_MMAX_A_Y", "FXI3SAB_VOR_COM_MM_A_Y", "FXI3SAB_VOR_COM_MMAX_A_Y", "NBJRR50_MM_A", "NBJRR50_MMAX_A", "NBJRR50_MSOM_A", "NBJRR1_MM_A", "NBJRR1_MMAX_A", "NBJRR1_MSOM_A", "NBJRR5_MM_A", "NBJRR5_MMAX_A", "NBJRR5_MSOM_A", "NBJRR10_MM_A", "NBJRR10_MMAX_A", "NBJRR10_MSOM_A", "NBJRR30_MM_A", "NBJRR30_MMAX_A", "NBJRR30_MSOM_A", "NBJRR100_MM_A", "NBJRR100_MMAX_A", "NBJRR100_MSOM_A", "RR_VOR_MM_A", "RR_VOR_MMAX_A", "RRAB_VOR_MM_A", "RRAB_VOR_MMAX_A", "TAILLE1", "TAILLE2",
]
ordinal_columns = {
    col: list(data[col].value_counts().sort_index().index)
    for col in ordinals
    if col in data.columns
}
ordinal_columns["PROPORTION_32"] += ["10. > 90"]
ordinal_columns.update(
    {
        "CARACT4": [
            "absence de surface", "Surface de moins d", "Surface entre 501", "Surface entre 1001", "Surface entre 1501", "Surface de plus de",
        ],
        "SURFACE4": [
            "0", "500", "1000", "1500", "2000", "2500", "3000", "3500", "4000", "4500", "5000", "5500", "6000", "6500", "7000", "7000+",
        ],
        "SURFACE6": [
            "0", "500", "1000", "1500", "2000", "2500", "3000", "3500", "4000", "4500", "5000", "5500", "6000", "6500", "7000", "7000+",
        ],
        "total_surface_2023": [
            "Aucun feu", "<10ha", "10-20ha", "20-50ha", "50-100ha", "100-200ha", ">200ha",
        ],
        "total_surface_5y": [
            "Aucun feu", "<10ha", "10-20ha", "20-50ha", "50-100ha", "100-200ha", ">200ha",
        ],
        "surface_over_forest": [
            "Absence de feu", "<0.05", "0.05-0.1", "0.1-0.2", "0.5-2",
        ],
        "fire_extinction_rates": ["Aucun feu", "<50%", "50-70%", "70-85%", ">85%"],
    }
)

# columns that are to be removed (target + no values)
to_remove = target.columns.tolist() + [target_col]
to_remove += [c for c in data.columns if "MMSOM" in c]
to_remove += ["DEROG13", "DEROG14", "DEROG16"]

# removing columns
categorical_columns = [
    col.name
    for col in features.categoricals
    if col not in to_remove and col not in ordinal_columns
]
categorical_columns += ["TYPERS"]
numerical_columns = [
    col.name
    for col in features.numericals
    if col not in to_remove
    and col not in ordinal_columns
    and col not in categorical_columns
]
print(
    len(categorical_columns),
    len(numerical_columns),
    len(ordinal_columns),
    len(categorical_columns) + len(numerical_columns) + len(ordinal_columns),
)

197 120 238 555


## 2026 carving — OrdinalCarver + multiprocessing + Wilson-CI

`n_jobs` parallelises the per-feature combination search; set it to the core
count on the competition machine. `min_freq_alpha=0.05` is the 95% Wilson
interval used to test each bin's frequency.


In [6]:
import time

from AutoCarver import Features, OrdinalCarver
from AutoCarver.discretizers import ProcessingConfig

N_JOBS = 6

# the 0/1/2+ claim-count target is ordinal -> carve with Kendall's Tau-c
y_ord_train = collapse_count(y_train)
y_ord_dev = collapse_count(y_dev)

config = ProcessingConfig(
    dropna=False, copy=False, verbose=False, n_jobs=N_JOBS, min_freq_alpha=0.05
)

### Qualitative + ordinal features

In [7]:
qualitatives = Features(categoricals=categorical_columns, ordinals=ordinal_columns)

carver = OrdinalCarver(features=qualitatives, target_scale="level", config=config)

t0 = time.perf_counter()
x_train = carver.fit_transform(x_train, y_ord_train, X_dev=x_dev, y_dev=y_ord_dev)
print(
    f"[2026] qualitative carving: {time.perf_counter() - t0:.1f}s on {N_JOBS} workers"
)
carver.summary

[OrdinalCarver] Carving:   0%|          | 0/435 [00:00<?, ?feature/s]

[OrdinalCarver] dropped 38/435 feature(s) (no robust train/dev combination): LOG_VETUSTE_REGION, MEN_num, IND_num, IND_0_Y1_num, IND_Y1_Y2_num, IND_Y4_Y5_num, IND_Y6_Y7_num, IND_Y7_Y8_num, IND_INC_num, IND_0_Y1_IND, IND_Y1_Y2_IND, IND_Y2_Y3_IND, IND_Y6_Y7_IND, IND_Y7_Y8_IND, IND_INC_IND, IND_INC, IND_0_Y1, IND, LOG_INC, MEN, PROPORTION_13, PROPORTION_12, PROPORTION_14, PROPORTION_11, PROPORTION_33, PROPORTION_41, PROPORTION_42, PROPORTION_51, PROPORTION_52, IND_Y1_Y2, IND_Y2_Y3, LOG_A1_A2, IND_Y4_Y5, IND_Y6_Y7, IND_Y7_Y8, LOG_APA3_num_LOG_TOT, LOG_AVA1_num_LOG_TOT, LOG_VETUSTE


[2026] qualitative carving: 120.0s on 6 workers


content  \
feature                    tau_b    tau_c    somersd  n_mod label                                                                   
Categorical('ACTIVIT2')    0.020905 0.002217 0.003905 4.0   0                                                        [ACT3, ACT2]   
                                                            1                           [ACT8, ACT9, ACT4, ACT6, ACT7, __OTHER__]   
                                                            2                                                                ACT1   
                                                            3                                                                ACT5   
Categorical('VOCATION')    0.042972 0.006714 0.007263 2.0   0                   [VOC1, VOC7, VOC5, VOC3, VOC2, __OTHER__, VOC4...   
...                                                                                                                           ...   
Categorical('LOG_VETUSTE') NaN      NaN      NaN      NaN   57.926829268292686                                 57.926829268292686   
                                                            64.02439024390245                                   64.02439024390245   
                                                            52.083333333333336                                 52.083333333333336   
                                                            43.292682926829265                                 43.292682926829265   
                                                            69.63855421686748                                   69.63855421686748   

                                                                                target_mean_level  \
feature                    tau_b    tau_c    somersd  n_mod label                                   
Categorical('ACTIVIT2')    0.020905 0.002217 0.003905 4.0   0                            0.002030   
                                                            1                            0.005593   
                                                            2                            0.006539   
                                                            3                            0.010893   
Categorical('VOCATION')    0.042972 0.006714 0.007263 2.0   0                            0.002050   
...                                                                                           ...   
Categorical('LOG_VETUSTE') NaN      NaN      NaN      NaN   57.926829268292686           0.007920   
                                                            64.02439024390245            0.008269   
                                                            52.083333333333336           0.008765   
                                                            43.292682926829265           0.008900   
                                                            69.63855421686748            0.008993   

                                                                                frequency  \
feature                    tau_b    tau_c    somersd  n_mod label                           
Categorical('ACTIVIT2')    0.020905 0.002217 0.003905 4.0   0                    0.051364   
                                                            1                    0.035537   
                                                            2                    0.773396   
                                                            3                    0.139702   
Categorical('VOCATION')    0.042972 0.006714 0.007263 2.0   0                    0.362458   
...                                                                                   ...   
Categorical('LOG_VETUSTE') NaN      NaN      NaN      NaN   57.926829268292686   0.020571   
                                                            64.02439024390245    0.048470   
                                                            52.083333333333336   0.020076   
                                                       

### Apply carver to dev + OOS

In [8]:
x_dev = carver.transform(x_dev)

data_path = "../data/"
oos = pd.read_csv(data_path + "test_input_5qJzHrr.csv", low_memory=False)
oos = proc.transform(oos)
oos = carver.transform(oos)

No processing for continuous features

In [9]:
quantitatives = Features(numericals=numerical_columns)

## Feature selection  *(single selector, one config)*

One `ClassificationSelector` over **both** feature types with a single `SelectionConfig`,
instead of one selector per type. `n_best_features` is a **total** budget, split **evenly**
across feature types (redistributing seats a type cannot fill), so the 100 below buys 50
carved qualitative + 50 raw quantitative features.

The split matters more than the total: on a 0.69 % positive-rate target, tilting the same
budget towards carved qualitatives measurably costs log loss, because each extra carved
feature adds up to 5 mostly-empty buckets to a rare-event stump ensemble. Respect the
pinned dependency floor -- older releases apportion the budget differently and silently
produce a different feature mix.

Filters are the current library defaults plus a redundancy cut (Cramer's V 0.9 on
qualitatives, Spearman 0.9 on quantitatives). The 2025 association thresholds are **not**
reproduced here; that is one of the stated deviations between the two eras.

In [10]:
from AutoCarver.selectors import (
    ClassificationSelector,
    CramervFilter,
    SelectionConfig,
    SpearmanFilter,
)

config = SelectionConfig(
    qualitative_filters=[CramervFilter(threshold=0.9)],
    quantitative_filters=[SpearmanFilter(threshold=0.9)],
)
selector = ClassificationSelector(
    qualitatives + quantitatives, n_best_features=100, config=config
)
selector.fit(x_train, y_ord_train)
print("selected:", len(selector.selected_features))
selector.summary

selected: 100


,feature,Nan,Mode,measure,association,rank,filter,redundancy,redundancy_with,selected
0,Numerical('KAPITAL32'),0.000000,0.347971,KruskalEtaSquared,0.003341,0.0,Spearman,0.000000,itself,True
1,Numerical('KAPITAL_MAX'),0.000000,0.248687,KruskalEtaSquared,0.002912,1.0,Spearman,0.824113,KAPITAL32,True
2,Numerical('SURFACE10'),0.019235,0.523992,KruskalEtaSquared,0.002625,2.0,Spearman,0.611342,KAPITAL32,True
3,Numerical('SURFACE1'),0.000000,0.071573,KruskalEtaSquared,0.002570,3.0,Spearman,0.640521,KAPITAL32,True
4,Numerical('SURFACE7'),0.010479,0.559276,KruskalEtaSquared,0.002430,4.0,Spearman,0.621579,SURFACE10,True
...,...,...,...,...,...,...,...,...,...,...
511,Ordinal('SURFACE4'),0.000000,0.605064,Tschuprowt,0.040019,NaN,Cramerv,0.999993,SURFACE6,False
512,Ordinal('total_surface_2023'),0.000000,0.587276,Tschuprowt,0.003196,NaN,None,NaN,None,False
513,Ordinal('total_surface_5y'),0.000000,0.731602,Tschuprowt,0.008750,NaN,None,NaN,None,False
514,Ordinal('surface_over_forest'),0.000000,0.842874,Tschuprowt,0.003599,NaN,None,NaN,None,False


In [11]:
pd.set_option("display.max_rows", None)
selector.summary

,feature,Nan,Mode,measure,association,rank,filter,redundancy,redundancy_with,selected
0,Numerical('KAPITAL32'),0.000000,0.347971,KruskalEtaSquared,3.341446e-03,0.0,Spearman,0.000000,itself,True
1,Numerical('KAPITAL_MAX'),0.000000,0.248687,KruskalEtaSquared,2.912102e-03,1.0,Spearman,0.824113,KAPITAL32,True
2,Numerical('SURFACE10'),0.019235,0.523992,KruskalEtaSquared,2.625000e-03,2.0,Spearman,0.611342,KAPITAL32,True
3,Numerical('SURFACE1'),0.000000,0.071573,KruskalEtaSquared,2.569790e-03,3.0,Spearman,0.640521,KAPITAL32,True
4,Numerical('SURFACE7'),0.010479,0.559276,KruskalEtaSquared,2.429846e-03,4.0,Spearman,0.621579,SURFACE10,True
5,Numerical('KAPITAL21'),0.014090,0.573636,KruskalEtaSquared,1.992941e-03,5.0,Spearman,0.706107,KAPITAL32,True
6,Numerical('NBBAT4'),0.000000,0.121360,KruskalEtaSquared,1.928590e-03,6.0,Spearman,0.834693,SURFACE1,True
7,Numerical('NBSINSTRT'),0.000000,0.727373,KruskalEtaSquared,1.636910e-03,7.0,Spearman,0.379864,KAPITAL32,True
8,Numerical('KAPITAL23'),0.003050,0.834321,KruskalEtaSquared,1.421355e-03,8.0,Spearman,0.422755,SURFACE7,True
9,Numerical('EQUIPEMENT6'),0.000000,0.100607,KruskalEtaSquared,1.197528e-03,9.0,Spearman,0.394208,KAPITAL21,True


In [12]:
best_features = selector.selected_features.names

## XGBoost + Optuna  *(unchanged objectives from `utils.objectives`)*

The model search is the same as 2025 so any dev-metric delta is attributable to
the carving step, not the tuner. Drop `N_TRIALS` while iterating.


In [13]:
import optuna

from utils.objectives import get_multiclass_objective

N_TRIALS = 300

objective = get_multiclass_objective(
    x_train[best_features],
    y_ord_train,
    x_dev[best_features],
    y_ord_dev,
    w_train=w_train,
    w_dev=w_dev,
)
# seeded so the tuning is reproducible: the split and every XGBoost estimator
# already use random_state=42, and carving is deterministic, so this is the last
# source of run-to-run variance.
study = optuna.create_study(
    direction="minimize", sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(objective, n_trials=N_TRIALS)
study.best_params

[I 2026-08-31 08:27:13,084] A new study created in memory with name: no-name-9480a73f-5280-4ead-adf6-bebffda6d764


C:\Users\defra\Desktop\git\PROJECTS\caa-challenge\.venv\Lib\site-packages\xgboost\core.py:751: UserWarning: [08:27:21] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[I 2026-08-31 08:27:22,489] Trial 0 finished with value: 11.439976647085183 and parameters: {'objective': 'multi:softprob', 'n_estimators': 287, 'learning_rate': 0.9507192349792751, 'max_depth': 8, 'min_child_weight': 12, 'subsample': 0.3248149123539492, 'colsample_bytree': 0.32479561626896214, 'colsample_bylevel': 0.24646688973455957, 'gamma': 8.661761457749352, 'alpha': 6.011150117432088, 'lambda': 7.080725777960454}. Best is trial 0 with value: 11.439976647085183.


[I 2026-08-31 08:27:27,128] Trial 1 finished with value: 10.12497945384667 and parameters: {'objective': 'multi:softprob', 'n_estimators': 110, 'learning_rate': 0.9699128611767781, 'max_depth': 9, 'min_child_weight': 5, 'subsample': 0.3454599737656805, 'colsample_bytree': 0.34672360788274703, 'colsample_bylevel': 0.4433937943676302, 'gamma': 5.247564316322379, 'alpha': 4.319450186421157, 'lambda': 2.9122914019804194}. Best is trial 1 with value: 10.12497945384667.


[I 2026-08-31 08:27:39,780] Trial 2 finished with value: 1.6465574086802413 and parameters: {'objective': 'multi:softprob', 'n_estimators': 406, 'learning_rate': 0.13957991126597663, 'max_depth': 3, 'min_child_weight': 8, 'subsample': 0.5648559873736287, 'colsample_bytree': 0.8281407691144109, 'colsample_bylevel': 0.3597390257266878, 'gamma': 5.142344384136116, 'alpha': 5.924145688620425, 'lambda': 0.46450412719997725}. Best is trial 2 with value: 1.6465574086802413.


...

[I 2026-08-31 09:06:00,587] Trial 296 finished with value: 0.940700757879865 and parameters: {'objective': 'multi:softprob', 'n_estimators': 233, 'learning_rate': 0.049320600661835454, 'max_depth': 2, 'min_child_weight': 11, 'subsample': 0.8116902041021831, 'colsample_bytree': 0.49185866603227346, 'colsample_bylevel': 0.5660958994612664, 'gamma': 8.608232088187506, 'alpha': 2.8285831588924304, 'lambda': 1.5773969992047154}. Best is trial 200 with value: 0.9199100734725815.


[I 2026-08-31 09:06:07,114] Trial 297 finished with value: 0.9427259175639068 and parameters: {'objective': 'multi:softprob', 'n_estimators': 211, 'learning_rate': 0.13863251963552914, 'max_depth': 1, 'min_child_weight': 8, 'subsample': 0.7372630407694911, 'colsample_bytree': 0.39143464644206, 'colsample_bylevel': 0.29121669364275793, 'gamma': 7.722169050118028, 'alpha': 0.8732676440280366, 'lambda': 2.789646148030148}. Best is trial 200 with value: 0.9199100734725815.


[I 2026-08-31 09:06:13,299] Trial 298 finished with value: 0.9401014551059885 and parameters: {'objective': 'multi:softprob', 'n_estimators': 142, 'learning_rate': 0.07767465411802069, 'max_depth': 2, 'min_child_weight': 18, 'subsample': 0.8348984823768036, 'colsample_bytree': 0.45959416394189095, 'colsample_bylevel': 0.5443470210520026, 'gamma': 8.154095892819353, 'alpha': 3.314752769394603, 'lambda': 1.1443864302075002}. Best is trial 200 with value: 0.9199100734725815.


[I 2026-08-31 09:06:18,250] Trial 299 finished with value: 0.9382513591656048 and parameters: {'objective': 'multi:softprob', 'n_estimators': 111, 'learning_rate': 0.10879655064021972, 'max_depth': 1, 'min_child_weight': 17, 'subsample': 0.798474782661936, 'colsample_bytree': 0.5997017230476283, 'colsample_bylevel': 0.5894879496514682, 'gamma': 7.384532027671357, 'alpha': 9.19389317846425, 'lambda': 2.4253190802419327}. Best is trial 200 with value: 0.9199100734725815.


{'objective': 'multi:softprob',
 'n_estimators': 109,
 'learning_rate': 0.07854518144874091,
 'max_depth': 2,
 'min_child_weight': 17,
 'subsample': 0.8218094537436436,
 'colsample_bytree': 0.46189056451948923,
 'colsample_bylevel': 0.5315954872997546,
 'gamma': 7.480450837106385,
 'alpha': 1.9495990790032933,
 'lambda': 1.9602840460106754}

In [14]:
from utils.objectives import get_best_multiclass_model

# 7.0.5-era helper, unchanged: prints Log Loss Train / Dev (the article's dev metric)
selected_features, model = get_best_multiclass_model(
    study.best_params,
    x_train[best_features],
    y_ord_train,
    x_dev[best_features],
    y_ord_dev,
    w_train=w_train,
    w_dev=w_dev,
    train_on_full=False,
)

used features: ['VOCATION', 'KAPITAL40', 'TYPERS', 'KAPITAL41', 'KAPITAL42', 'DEROG12', 'RISK11', 'KAPITAL43', 'KAPITAL37', 'DEROG4', 'ZONE', 'RISK10', 'DEROG5', 'AN_EXERC', 'total_surface_crossed', 'ACTIVIT2', 'TXAB_VOR_MM_A_TXAB_VOR_MMAX_A', 'NBJFXI3S10_MM_A_NBJFXI3S10_MMAX_A', 'INDEM2', 'EQUIPEMENT2', 'NBJFXI3S16_MM_A_NBJFXI3S16_MMAX_A', 'NBJTX0_MM_A_NBJTX0_MMAX_A', 'NBJRR50_MM_A_NBJRR50_MMAX_A', 'NBJRR10_MM_A_NBJRR10_MMAX_A', 'NBJFXI3S28_MM_A_NBJFXI3S28_MMAX_A', 'IND_SNV_REGION', 'NBJFF16_MM_A_NBJFF16_MMAX_A', 'RRAB_VOR_MM_A_RRAB_VOR_MMAX_A', 'NBJFF10_MM_A_NBJFF10_MMAX_A', 'NBJFXY8_MM_A_NBJFXY8_MMAX_A', 'NBJRR30_MM_A_NBJRR30_MMAX_A', 'TAMPLIAB_VOR_MM_A_TAMPLIAB_VOR_MMAX_A', 'TAMPLIM_VOR_MM_A_TAMPLIM_VOR_MMAX_A', 'ALTITUDE_5_ALT_TOT', 'SURFACE6', 'TAILLE1', 'TAILLE2', 'HAUTEUR_MAX', 'BDTOPO_BAT_MAX_HAUTEUR_MAX', 'BDTOPO_BAT_MAX_HAUTEUR', 'DISTANCE_213', 'PROPORTION_21', 'HAUTEUR', 'CARACT4', 'DISTANCE_212', 'DISTANCE_411', 'DISTANCE_323', 'DISTANCE_335', 'DISTANCE_334', 'NBJRR10_MSO

Log Loss Train: 0.7819988979524337


Log Loss Dev:   0.9199100734725815


### Hand-off to `amount_model_2026.ipynb` — train + dev

The severity model weights each observation by the **absolute error of the
frequency model** (`|pred_sum - FREQ|`), so it needs this model's predictions on
train, dev and OOS. Written with the index (`ID`), as in 2025.

In [15]:
# hand-off to the severity model: train + dev frequency predictions
# (mirrors the 2025 notebook, which wrote frequency_012.csv)
FREQ_VERSION = "2026"
CLASS_MEANS = np.array([0, 1, 2.076923076923077])


def frequency_predictions(frame):
    """P(1), P(2+) and the expected claim count, built as one block.

    Assigning the three columns one at a time inserts into a ~560-column frame
    three times over and pandas warns that the result is highly fragmented; one
    concat is both quieter and cheaper.
    """
    proba = model.predict_proba(frame[selected_features])
    return pd.DataFrame(
        {
            "pred_1": proba[:, 1],
            "pred_2": proba[:, 2],
            "pred_sum": (proba * CLASS_MEANS).sum(axis=1),
        },
        index=frame.index,
    )


def with_predictions(frame):
    """Attach the block, replacing the columns if the cell is re-run."""
    block = frequency_predictions(frame)
    return pd.concat(
        [frame.drop(columns=block.columns, errors="ignore"), block], axis=1
    )


x_train = with_predictions(x_train)
x_dev = with_predictions(x_dev)

merged = pd.concat([x_train, x_dev], axis=0)
merged.to_csv(data_path + f"frequency_{FREQ_VERSION}.csv")
print("wrote", data_path + f"frequency_{FREQ_VERSION}.csv", merged.shape)

wrote ../data/frequency_2026.csv (383610, 566)


## Prediction — law of total expectation

In [16]:
# E[Y] = P(1)*E[Y|1] + P(2+)*E[Y|2+]; second-class mean carried over from 2025
oos_proba = model.predict_proba(oos[selected_features])
oos_expected_count = (oos_proba * CLASS_MEANS).sum(axis=1)

# one block, one concat -- same reason as the train/dev hand-off above.
# FREQ and pred_sum are the same quantity: 2025 named it FREQ for the submission
# and pred_sum for the severity hand-off, and both names are consumed downstream.
oos_block = pd.DataFrame(
    {
        "FREQ": oos_expected_count,
        "pred_1": oos_proba[:, 1],
        "pred_2": oos_proba[:, 2],
        "pred_sum": oos_expected_count,
    },
    index=oos.index,
)
oos = pd.concat(
    [oos.drop(columns=oos_block.columns, errors="ignore"), oos_block], axis=1
)

pred = oos[["ID", "ANNEE_ASSURANCE", "FREQ"]]
# pred.to_csv("predictions/2026_pred.csv", index=False)
pred.head()

,ID,ANNEE_ASSURANCE,FREQ
0,383611,0.813699,0.963831
1,383612,1.000000,0.694419
2,383613,0.586301,0.400150
3,383614,1.000000,0.373611
4,383615,0.753425,0.495960


### Hand-off to `amount_model_2026.ipynb` — OOS

In [17]:
# hand-off to the severity model: OOS frequency predictions
# (mirrors the 2025 notebook, which wrote oos_frequency_012.csv)
# pred_1 / pred_2 / pred_sum were built with FREQ in the cell above, in one block
oos.to_csv(data_path + f"oos_frequency_{FREQ_VERSION}.csv", index=False)
print("wrote", data_path + f"oos_frequency_{FREQ_VERSION}.csv", oos.shape)

wrote ../data/oos_frequency_2026.csv (95852, 564)


### Target encoding for the severity model  *(re-added from 2025)*

`amount_model_2026.ipynb` stratifies its train/dev split on a **discretized `CM`**, and
that discretization is itself a carving: **feature = `CM` (severity), target = frequency**.
2025 built it here, in the frequency notebook (`frequency_model.ipynb` cell 61), and the
severity notebook only ever *loaded* the artifact.

It lives here rather than in the severity notebook for a reason: the recipe fits on
`x_train` / `x_dev`, so putting it in the severity notebook would need a split that does
not exist yet -- the split it is meant to stratify. Fitting it here breaks the circle.

Recipe, unchanged from 2025:

* fit on rows that **actually have a claim** (`y > 0`), not the full frame -- otherwise
  the 99.4 % of rows with `CM == 0` swamp every bucket;
* binary target = **more than one claim** (`collapse_count(y) - 1`, so 0 = exactly one
  claim, 1 = two or more);
* `min_freq=0.05`, `max_n_mod=5`, `dropna=False`, Tschuprow's T (the 7.6 `BinaryCarver`
  default, matching the 2025 `..._tschuprowt` artifact).

Saved in **light mode** -- the 2025 artifact was 56 MB because it persisted the full
combination history; light mode drops it and lands at ~1.6 kB, small enough to commit.

In [18]:
from pathlib import Path

from AutoCarver import BinaryCarver

# feature = CM (severity), target = frequency (binary: more than one claim)
target_carver = BinaryCarver(
    features=Features(numericals=["CM"]),
    min_freq=0.05,
    max_n_mod=5,
    config=ProcessingConfig(dropna=False, copy=True, verbose=False),
)
target_carver.fit(
    x_train[y_train > 0],
    collapse_count(y_train)[y_train > 0] - 1,
    X_dev=x_dev[y_dev > 0],
    y_dev=collapse_count(y_dev)[y_dev > 0] - 1,
)

TARGET_CARVER_PATH = Path("model/cm_carver_freq_tschuprowt_2026.json")
target_carver.save(TARGET_CARVER_PATH, light_mode=True)
print(
    "wrote", TARGET_CARVER_PATH, f"({TARGET_CARVER_PATH.stat().st_size / 1024:.1f} kB)"
)
target_carver.summary

[BinaryCarver] Carving:   0%|          | 0/1 [00:00<?, ?feature/s]

wrote model\cm_carver_freq_tschuprowt_2026.json (1.7 kB)


content  \
feature         cramerv  tschuprowt n_mod label                         
Numerical('CM') 0.070898 0.050132   5     0          (-inf, 6.01e+02]   
                                          1      (6.01e+02, 8.69e+02]   
                                          2      (8.69e+02, 1.49e+03]   
                                          3      (1.49e+03, 5.14e+04]   
                                          4           (5.14e+04, inf)   

                                                 target_mean  frequency  \
feature         cramerv  tschuprowt n_mod label                           
Numerical('CM') 0.070898 0.050132   5     0         0.026403   0.445807   
                                          1         0.067961   0.050515   
                                          2         0.025974   0.075527   
                                          3         0.049475   0.327121   
                                          4         0.024272   0.101030   

                                                 count  dropped dropped_reason  
feature         cramerv  tschuprowt n_mod label                                 
Numerical('CM') 0.070898 0.050132   5     0      909.0    False           None  
                                          1      103.0    False           None  
                                          2      154.0    False           None  
                                          3      667.0    False           None  
                                          4      206.0    False           None